# Data analysis, session 2C: Run, collect, compare
**October 15, 11:15-noon (45 minutes).** 30 minutes on automation, then a 15-minute homework clinic.

| Time | Activity |
|---|---|
| 11:15-11:25 | Parameters, isolated kernels, one report |
| 11:25-11:35 | Loop, record failures, collect only successful runs |
| 11:35-11:45 | Summary table and comparison chart |
| 11:45-noon | Homework mapping, failure exercise, questions |

This module starts in a fresh kernel and does not require manually running the template first.
Default: **three stocks**, one period, no repeated AAPL run. A new output folder prevents stale reports entering the comparison.
The multi-period extension is opt-in below.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sqlite3
from pathlib import Path
from IPython.display import display
%matplotlib inline
course_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').is_file())
lesson_dir = course_root / 'Lectures/Lecture 4'
plt.style.use('seaborn-v0_8-whitegrid')
import papermill as pm
import scrapbook as sb
import os
import sys
import tempfile
import json
from contextlib import closing

## 1. A template is an executable document
Papermill injects parameter overrides after the tagged cell, starts an independent kernel, executes all cells, and saves the results.
Scrapbook embeds selected results inside those outputs so we can collect metrics without rerunning the analysis.
See the [Papermill execution reference](https://papermill.readthedocs.io/en/2.3.4/reference/papermill-workflow.html).
The small kernel setup below keeps child reports on the same Python environment as this notebook.

In [ ]:
template = lesson_dir / '2_stock_analysis_template.ipynb'
db_path = lesson_dir / 'data/data.db'
if not db_path.is_file():
    raise FileNotFoundError(f'Download the bCourses database to {db_path}')
with closing(sqlite3.connect(db_path.resolve().as_uri() + '?mode=ro', uri=True)) as conn:
    available = pd.read_sql_query('SELECT DISTINCT ticker FROM ohlc ORDER BY ticker', conn)['ticker'].tolist()
print('Available tickers:', available)
tickers = ['AAPL', 'MSFT', 'NVDA']
start_date, end_date = '2024-01-01', '2024-06-30'
output_root = lesson_dir / 'outputs'
output_root.mkdir(exist_ok=True)
run_dir = Path(tempfile.mkdtemp(prefix='reports_', dir=output_root))
# Register a kernel only inside this run's output directory (no user-level installation).
kernel_root = run_dir / 'jupyter'
kernel_dir = kernel_root / 'kernels' / 'course-report'
kernel_dir.mkdir(parents=True)
(kernel_dir / 'kernel.json').write_text(json.dumps({
    'argv': [sys.executable, '-m', 'ipykernel_launcher', '-f', '{connection_file}'],
    'display_name': 'Course report', 'language': 'python'}))
previous_jupyter_path = os.environ.get('JUPYTER_PATH')
os.environ['JUPYTER_PATH'] = str(kernel_root) + (os.pathsep + previous_jupyter_path if previous_jupyter_path else '')
print('Reports will be saved in:', run_dir)

In [ ]:
def run_report(ticker, start, end, folder):
    """Run one independent report; propagate errors so the caller can record failures."""
    folder = Path(folder)
    folder.mkdir(parents=True, exist_ok=True)
    output = folder / f'{ticker}_{start}_{end}.ipynb'
    pm.execute_notebook(
        str(template), str(output),
        parameters={'ticker': ticker, 'start_date': start, 'end_date': end,
                    'db_path': str(db_path.resolve()), 'volatility_window': 20},
        kernel_name='course-report', cwd=str(lesson_dir), progress_bar=False,
    )
    return output

first_report = run_report(tickers[0], start_date, end_date, run_dir)
first = sb.read_notebook(str(first_report))
print('First report:', first.scraps['ticker'].data, first.scraps['total_return_pct'].data)
print('Scraps:', list(first.scraps))

## 2. Keep a manifest of this run
Catch a failed ticker, record its error, and continue. A failed notebook may exist on disk; it must not enter the summary.
Collect the exact paths returned by successful runs instead of scanning an output folder that may contain older work.

In [ ]:
successful_paths = [first_report]
failures = []
for symbol in tickers[1:]:
    try:
        successful_paths.append(run_report(symbol, start_date, end_date, run_dir))
    except Exception as exc:
        failures.append({'ticker': symbol, 'error': str(exc)})
        print(f'FAILED: {symbol}: {exc}')
(run_dir / 'failed_stocks.json').write_text(json.dumps(failures, indent=2))
(run_dir / 'manifest.json').write_text(json.dumps({
    'start_date': start_date, 'end_date': end_date,
    'reports': [path.name for path in successful_paths], 'failures': failures}, indent=2))
print(f'Successfully generated {len(successful_paths)}/{len(tickers)} reports')

## 3. Collect results with Scrapbook
`sb.glue` in the template stores named values; `sb.read_notebook` reads them back.
Enforce unique `(ticker, start, end)` keys and consistent dates before comparing reports.

In [ ]:
def collect_reports(paths):
    """Build a summary from successful paths only; reject empty or duplicated input."""
    rows = []
    names = ['ticker', 'start_date', 'end_date', 'trading_days', 'total_return_pct',
             'avg_volatility', 'max_drawdown_pct', 'data_quality_ok']
    for path in paths:
        scraps = sb.read_notebook(str(path)).scraps
        row = {name: scraps[name].data for name in names}
        row['report_path'] = str(path)
        rows.append(row)
    if not rows:
        raise ValueError('No successful reports to summarize; inspect failed_stocks.json.')
    summary = pd.DataFrame(rows)
    if summary.duplicated(['ticker', 'start_date', 'end_date']).any():
        raise ValueError('Duplicate report keys in this run.')
    return summary.sort_values('total_return_pct', ascending=False).reset_index(drop=True)

summary = collect_reports(successful_paths)
assert summary['start_date'].eq(start_date).all() and summary['end_date'].eq(end_date).all()
assert summary['data_quality_ok'].all()
summary.to_csv(run_dir / 'portfolio_summary.csv', index=False)
display(summary.drop(columns='report_path'))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), layout='constrained')
summary.sort_values('total_return_pct').plot.barh(x='ticker', y='total_return_pct',
    ax=axes[0], legend=False, title='Total return over the common period', xlabel='Return (%)')
axes[1].scatter(summary['avg_volatility'] * 100, summary['total_return_pct'])
for row in summary.itertuples():
    axes[1].annotate(row.ticker, (row.avg_volatility * 100, row.total_return_pct),
                     xytext=(4, 4), textcoords='offset points')
axes[1].set(xlabel='Mean rolling volatility, annualized (%)', ylabel='Total return (%)',
            title='Return and variability')
fig.savefig(run_dir / 'comparison.png', dpi=150, bbox_inches='tight')
plt.show()
# Full data also travels with the report, if you need to audit a metric.
aapl_prices = first.scraps['prices'].data
assert np.isclose(first.scraps['total_return_pct'].data,
                  (aapl_prices['adj_close'].iloc[-1] / aapl_prices['adj_close'].iloc[0] - 1) * 100)
display(aapl_prices.tail(3))

## 4. Homework clinic and failure exercise (15 minutes)
[Homework 2](../Lecture%202/hw2.md), due October 29, still has three deliverables:

| Homework task | Starting point |
|---|---|
| Diagnose, clean, log and validate `ohlc_hw` | Module 2A + October 8 cleaning |
| Parameterized risk report | Module 2B; add the requested Sharpe ratio, metrics, and visual styling |
| Run every ticker over six complete months | This module; replace the three-stock demonstration and fixed dates |

**Try:** use a nonexistent ticker in a copy of the loop. Predict the error, verify it is logged, and confirm its incomplete notebook does not enter the summary.
**Discuss:** why must calendar dates, adjustment conventions, and return units match before comparing reports?
For homework's six-month window, anchor the period to the latest date in the database, not the computer's current date; exclude that date's unfinished month.
Keep the raw database local and never submit credentials.

In [ ]:
# Practice here: choose a nonexistent ticker and catch its report error.

## Optional extension: multiple periods
Outside the 180-minute session. Set `RUN_EXTENSIONS = True` to run three tickers over four periods (12 additional reports).
Use 2023, a complete year for these three stocks. Their final 2024 quarter has only 14 observations, which is too short for a 20-day return window even though other tickers extend later.
This preserves the original multi-period example without making students wait for it during the live lesson.

In [ ]:
RUN_EXTENSIONS = False
periods = [('2023-01-01', '2023-03-31', 'Q1'), ('2023-04-01', '2023-06-30', 'Q2'),
           ('2023-07-01', '2023-09-30', 'Q3'), ('2023-10-01', '2023-12-31', 'Q4')]
if RUN_EXTENSIONS:
    quarterly_paths, quarterly_failures = [], []
    for symbol in tickers:
        for start, end, label in periods:
            try:
                quarterly_paths.append(run_report(symbol, start, end, run_dir / 'quarterly'))
            except Exception as exc:
                quarterly_failures.append({'ticker': symbol, 'period': label, 'error': str(exc)})
    (run_dir / 'quarterly_failures.json').write_text(json.dumps(quarterly_failures, indent=2))
    quarterly = collect_reports(quarterly_paths)
    labels = {start: label for start, end, label in periods}
    quarterly['period'] = quarterly['start_date'].map(labels)
    display(quarterly.pivot(index='ticker', columns='period', values='total_return_pct'))
    display(quarterly.pivot(index='ticker', columns='period', values='avg_volatility'))
else:
    print('Multi-period extension not requested; live lesson is complete.')

In [ ]:
# Restore the notebook process environment after launching all child reports.
if previous_jupyter_path is None:
    os.environ.pop('JUPYTER_PATH', None)
else:
    os.environ['JUPYTER_PATH'] = previous_jupyter_path
print('Saved reports, manifest, failures, summary CSV, and comparison figure:', run_dir)

## Recap
Filter with SQL, validate rows, calculate clearly defined metrics, parameterize once, execute independently, and collect successful outputs.
You now have the complete workflow needed to build Homework 2. Optional details are linked from [the lesson guide](readme.md).